# Declare Packages

In [2]:
import pandas as pd
import numpy as np
from sklearn.utils import Bunch
from catboost import CatBoostClassifier,  Pool
# from geopy.distance import geodesic
import pickle

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
# warnings.filterwarnings("ignore", category=pd.core.generic.SettingWithCopyWarning)
warnings.filterwarnings("ignore")

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

from google.cloud import bigquery
from google.cloud import storage
import os
from datetime import datetime

path = r"C:\Users\Dwaipayan C\AppData\Roaming\gcloud\application_default_credentials.json"
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = path
client = bigquery.Client(project='prj-prod-dataplatform')
os.environ["GOOGLE_CLOUD_PROJECT"] = "prj-prod-dataplatform"

import sys
sys.path.append(r"D:\Officedata\Feature_Universe_Training")
from Utils.bq_fast_download import bq_to_df

# Config

In [ ]:
# B Score model configuration
# ─────────────────────────────────────────────────────────────────────────────
from google.cloud import bigquery
import time
import pandas as pd
from datetime import datetime

# client = bigquery.Client()
CURRENT_DATE = datetime.now().strftime("%Y%m%d")

# Schema and table name definitions (update OBDATE for each model run)
schema1 = 'risk_mart'
schema2 = 'worktable_data_analysis'
schema3 = 'dap_ds_poweruser_playground'

UTIL_PATH = f"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Model_Data_Set_preparation\DC_Model_Data_Sets_Feature_Universe\Feature_Universe_Training_Config"

# UTIL_PATH = F"D:\Officedata\Feature_Universe_Training"

GS_BUCKET = "prod-asia-southeast1-tonik-report-analytics"
GS_BUCKET_OLD = "prod-asia-southeast1-tonik-aiml-workspace"
PROJECT_ID = "prj-prod-dataplatform"

tab = 't_score_snapshot'
ver = 'v1'

OBDATE = '2026-10-27'   # Observation date — update for each scoring run
SDATE  = '2023-01-01'   # Start date for training window
EDATE  = OBDATE

formatted_date = datetime.strptime(SDATE, '%Y-%m-%d').strftime('%Y%m%d')
ob_formatted   = datetime.strptime(OBDATE, '%Y-%m-%d').strftime('%Y%m%d')

al       = f'applied_loans_20230101_{ob_formatted}'
altrans  = f'applied_loans_20210701_{ob_formatted}_trans'
nal      = f'tsa_onboarded_but_never_applied_loan_20230101_{ob_formatted}'

# Intermediate staging tables
bscorebasedata = f'{tab}_base_data_{ver}_{datetime.strptime(SDATE, '%Y-%m-%d').strftime('%Y%m%d')}_{datetime.strptime(OBDATE, '%Y-%m-%d').strftime('%Y%m%d')}'
print(bscorebasedata)
bscoresnapshotcustomerdata        = f'{tab}_applied_loans_customer_information_data_{ver}'
bscoresnapshotcustomertransactiondata = f'{tab}_applied_loans_customer_transaction_data_{ver}'
bscoresnapshotcustomerevent       = f'{tab}_applied_loans_customer_event_data_{ver}'
bscoresnapshotcontactability      = f'{tab}_applied_loans_contactability_{ver}'
bscoresnapshotcombineddata        = f'{tab}_model_applied_loans_{ver}_{ob_formatted}'


step61 = f'{schema2}.{tab}_vb_results_{ver}_{ob_formatted}'
step62 = f'{schema2}.{tab}_chatbot_{ver}_{ob_formatted}'
step63 = f'{schema2}.{tab}_chatbot_fb_{ver}_{ob_formatted}'
step64 = f'{schema2}.{tab}_chatbot_users_{ver}_{ob_formatted}'
step65 = f'{schema2}.{tab}_genesys_chat_{ver}_{ob_formatted}'
step66 = f'{schema2}.{tab}_genesys_email_{ver}_{ob_formatted}'
step67 = f'{schema2}.{tab}_inbound_voice_{ver}_{ob_formatted}'
step68 = f'{schema2}.{tab}_airudder_softcollection_calls_{ver}_{ob_formatted}'
step682 = f'{schema2}.{tab}_cs_vb_cnt_calls_total_{ver}_{ob_formatted}'
step69 = f'{schema2}.{tab}_reach_enhanced_{ver}_{ob_formatted}'
step610 = f'{schema2}.{tab}_ars_results_{ver}_{ob_formatted}'
step611 = f'{schema2}.{tab}_inbound_additional_features_{ver}_{ob_formatted}'
step612 = f'{schema2}.{tab}_outbound_voice_features_{ver}_{ob_formatted}'
step6 = f'{schema2}.{tab}_contact_features_extended_{ver}_{ob_formatted}'
step7 = f'{schema2}.{tab}_contact_genesysdata_{ver}_{ob_formatted}'
step8 = f'{schema2}.{tab}_loan_level_swathi_table_{ver}_{ob_formatted}'
step9 = f'{schema2}.{tab}_ptp_details_{ver}_{ob_formatted}'

step101 = f'{schema2}.{tab}_account_product_portfolio_composition_{ver}_{ob_formatted}'
step102 = f'{schema2}.{tab}_activity_recency_dormancy_{ver}_{ob_formatted}'
step103 = f'{schema2}.{tab}_tx_balance_liquidity_{ver}_{ob_formatted}'
step104 = f'{schema2}.{tab}_bureau_external_scoring_{ver}_{ob_formatted}'
step105 = f'{schema2}.{tab}_completion_settlement_metrics_{ver}_{ob_formatted}'
step106 = f'{schema2}.{tab}_conversion_campaign_engagement_events_{ver}_{ob_formatted}'
step107 = f'{schema2}.{tab}_customer_acquisition_retention_{ver}_{ob_formatted}'
step108 = f'{schema2}.{tab}_customer_lifecycle_engagement_{ver}_{ob_formatted}'
step109 = f'{schema2}.{tab}_demographic_verification_enhanced_{ver}_{ob_formatted}'
step1010 = f'{schema2}.{tab}_deposit_savings_products_{ver}_{ob_formatted}'
step1011 = f'{schema2}.{tab}_derived_ratio_metrics_{ver}_{ob_formatted}'
step1012 = f'{schema2}.{tab}_fraud_compliance_indicators_{ver}_{ob_formatted}'
step1013 = f'{schema2}.{tab}_loan_product_specific_metrics_{ver}_{ob_formatted}'
step1014 = f'{schema2}.{tab}_loan_staging_milestones_{ver}_{ob_formatted}'
step10151 = f'{schema2}.{tab}_payment_behaviour_advanced_{ver}_{ob_formatted}'
step10152 = f'{schema2}.{tab}_payment_behaviour_advanced_2_{ver}_{ob_formatted}'
step1016 = f'{schema2}.{tab}_revenue_fee_decomposition_{ver}_{ob_formatted}'
step1017 = f'{schema2}.{tab}_risk_segmentation_categorization_{ver}_{ob_formatted}'
step1018 = f'{schema2}.{tab}_time_window_cohort_features_{ver}_{ob_formatted}'
step1019 = f'{schema2}.{tab}_transaction_volume_type_breakdown_{ver}_{ob_formatted}'
step111 = f'{schema2}.{tab}_collection_call_features_{ver}_{ob_formatted}'
step112 = f'{schema2}.{tab}_payhinga_features_{ver}_{ob_formatted}'

step12c = f'{schema2}.PH_City_Province_Event_Mapping_{ver}_{ob_formatted}'

step12d = f'{schema2}.{tab}_ph_shock_feature_{ver}_{ob_formatted}'
step14 = f'{schema2}.{tab}_prevloan_less_6dpd_details_{ver}_{ob_formatted}'

step17 = f'{schema2}.{tab}_customer_level_5dpd_details_{ver}_{ob_formatted}'

step20 = f'dap_ds_poweruser_playground.{tab}_yuvashri_features_{ver}_{ob_formatted}'

step22 = f'{schema2}.{tab}_finscore_{ver}_{ob_formatted}'
step23 = f'{schema2}.{tab}_trustingsocialscore_{ver}_{ob_formatted}'
step24 = f'{schema2}.{tab}_bureauinc_score_{ver}_{ob_formatted}'

for step_name, step_value in {
    "step61": step61,
    "step62": step62,
    "step63": step63,
    "step64": step64,
    "step65": step65,
    "step66": step66,
    "step67": step67,
    "step68": step68,
    "step682": step682,
    "step69": step69,
    "step610": step610,
    "step611": step611,
    "step612": step612,
    "step6": step6,
    "step7": step7,
    "step8": step8,
    "step9": step9,
    "step101": step101,
    "step102": step102,
    "step103": step103,
    "step104": step104,
    "step105": step105,
    "step106": step106,
    "step107": step107,
    "step108": step108,
    "step109": step109,
    "step1010": step1010,
    "step1011": step1011,
    "step1012": step1012,
    "step1013": step1013,
    "step1014": step1014,
    "step10151": step10151,
    "step10152": step10152,
    "step1016": step1016,
    "step1017": step1017,
    "step1018": step1018,
    "step1019": step1019,
    "step111": step111,
    "step112": step112,
    "step12c": step12c,
    "step12d": step12d,
    "step14": step14,
    "step17": step17,
    "step20": step20,
    "step22": step22,
    "step23": step23,
    "step24": step24,
}.items():
    print(f"{step_name}: {step_value}")



# Final output tables (split by loan type)
trench3_b_score_model_applied_loans = f'{tab}_model_applied_loans_app_package_{ver}_{ob_formatted}'

print(f"OBDATE: {OBDATE}")
print(f"Output table: {schema2}.{bscoresnapshotcombineddata}")

print(f"Applied Table: {schema1}.{al}")
print(f"Applied Trans Table: {schema1}.{altrans}")
print(f"tsa_onboarded_but_never_applied_loan Table: {schema1}.{nal}")

APP_PACKAGE = f'{tab}_applied_all_dis_loan_app_package_{ver}_{ob_formatted}'
print(f"App Package Table: {schema2}.{APP_PACKAGE}")

App_Feature_Table = f'{tab}_model_app_features_{ver}_{ob_formatted}'
print(f"App Feature Table: {schema2}.{App_Feature_Table}")

cic_feature_table = f'{tab}_cic_features_{ver}_{ob_formatted}'
print(f"CIC Feature Table: {schema2}.{tab}_cic_features_{ver}_{ob_formatted}")

android_devise_score_features = f"{tab}_android_device_score_feature_list_{ver}_{ob_formatted}"
print(f"Android Device Score Feature Table: {schema2}.{android_devise_score_features}")

ios_device_score_features = f"{tab}_ios_device_score_features_{ver}_{ob_formatted}"
print(f"iOS Device Score Feature Table: {schema2}.{ios_device_score_features}")

nitin_event_features = f"{schema2}.{tab}_events_feats_{ver}_{ob_formatted}"
print(f"nitin_event_features Table: {nitin_event_features}")

demo_features = f"{schema3}.{tab}_demo_features_{ver}"
print(f"Demo Feature Table: {demo_features}")
stepfinaltab = f"{schema2}.{tab}_model_full_features_v7_{ver}_{ob_formatted}"
print(f"Final Feature Table: {stepfinaltab}")

finaltab = f"{schema2}.{tab}_model_full_features_model_dataset_{ver}_{ob_formatted}"
print(f"Final Table: {finaltab}")

In [4]:
sq = f"""
create or replace table worktable_data_analysis.applied_all_dis_credo_device_info as
(WITH base_data AS (
  SELECT *
  FROM {schema2}.{bscorebasedata}
  WHERE DATE(ln_loan_appln_time) BETWEEN '2024-09-01' AND '{OBDATE}'
),
credolab_track AS (
  SELECT
    CAST(B.cust_id AS INT64) AS customerid,
    A.refno,
    datetime(A.createdOn) createdOn
  FROM `dl_loans_db_raw.tdbk_credolab_track` A
  JOIN `dl_customers_db_raw.tdbk_customer_mtb` B
    ON A.userId = B.user_id
),
credolab_android AS (
  SELECT
    A.customerid,
    B.referenceNumber AS refno,
    B.requestedDate AS createdOn
  FROM `risk_credit_mis.loan_master_table` A
  JOIN (
    SELECT
      digitalLoanAccountId,
      referenceNumber,
      requestedDate
    FROM `credolab_raw.credolab_insights_android`
    WHERE referenceNumber NOT IN (
      SELECT refno FROM `dl_loans_db_raw.tdbk_credolab_track`
    )
  ) B
  ON A.digitalLoanAccountId = B.digitalLoanAccountId
),
insight_dl AS (
  SELECT * FROM credolab_track
  UNION ALL
  SELECT * FROM credolab_android
)
SELECT DISTINCT
  A.customerid,
  loan.digitalLoanAccountId,
  insight_dl.refno,
  TIMESTAMP(insight_dl.createdOn, 'Asia/Manila') AS credo_inquiry_date_Manila,
  A.ln_loan_appln_time,Observation_date
FROM base_data A
JOIN `risk_credit_mis.loan_master_table` loan
  ON A.digitalLoanAccountId = loan.digitalLoanAccountId
LEFT JOIN insight_dl
  ON loan.customerId = insight_dl.customerid
WHERE DATE(insight_dl.createdOn) <= A.Observation_date
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY loan.digitalLoanAccountId
  ORDER BY insight_dl.createdOn DESC
) = 1)
;
"""
job = client.query(sq)
job.result()  # Wait for the job to complete.
print(f'Table worktable_data_analysis.applied_all_dis_credo_device_info created successfully')
print("Total columns:", client.get_table(f"worktable_data_analysis.applied_all_dis_credo_device_info").schema.__len__())

table = client.get_table(f"worktable_data_analysis.applied_all_dis_credo_device_info")

print("Total columns:", len(table.schema))
print("Total rows:", table.num_rows)

Table worktable_data_analysis.applied_all_dis_credo_device_info created successfully
Total columns: 6
Total columns: 6
Total rows: 221542


In [5]:
andr_query = f""" 
SELECT
  ios.*,
  dev.*
FROM `credolab_raw.android_credolab_datasets_struct_columns` AS ios

INNER JOIN `worktable_data_analysis.applied_all_dis_credo_device_info` AS dev
ON ios.deviceId = dev.refno
 
  """

# Run the query and save results to a Pandas DataFrame
# df_andr = client.query(andr_query).to_dataframe(progress_bar_type='tqdm')
df_andr = bq_to_df(andr_query)
print(f"The shape of the dataframe is: {df_andr.shape}")



INFO:Utils.bq_fast_download:Running query (anonymous destination, no named table)
INFO:Utils.bq_fast_download:Query finished. Anonymous result table: prj-prod-dataplatform._72abd532fd87b7ae4e292515574e1092acf0bf59.anona44c595ab686b3767c9e165da6a6376855dfb4136f7a620d8387ce5ae64b57aa
INFO:Utils.bq_fast_download:Extracting results to gs://prod-asia-southeast1-tonik-aiml-workspace/DC/Parquet_files/61a92f6dfb3e/tmp_result-*.parquet
INFO:Utils.bq_fast_download:Downloading 1 parquet shard(s) with 16 workers
INFO:Utils.bq_fast_download:Combining 1 local parquet file(s) via DuckDB
INFO:Utils.bq_fast_download:Loaded DataFrame with shape (165989, 14)
INFO:Utils.bq_fast_download:Deleted 1 GCS parquet file(s)


The shape of the dataframe is: (165989, 14)


In [6]:
df_andr.head(2)

,deviceId,GeneralInfo,Hardware,GeneralData,PhoneSettings,Network,HealthStats,run_date,customerid,digitalLoanAccountId,refno,credo_inquiry_date_Manila,ln_loan_appln_time,Observation_date
0,af643030-48f4-4355-86b8-628c2fceaa00,"{'file_structure': 9, 'mobile_platform': 0, 's...","{'device__brand': 'CONSUNG', 'device__model': ...","{'telephony_info__network_operator_name': '', ...","{'accelerometer_rotation': 1, 'airplane_mode_o...","{'capability_mms': 0, 'capability_congested': ...","{'data_type': 'UidHealthStats', 'measurement_k...",2025-06-26,2974277,3df672d8-731a-4e1d-a96a-442619585319,af643030-48f4-4355-86b8-628c2fceaa00,2025-06-26 00:23:10,2025-06-27 14:09:49,2025-06-26
1,6d776152-1d52-42c9-9dd3-bef7df31ed0e,"{'file_structure': 9, 'mobile_platform': 0, 's...","{'device__brand': 'HONOR', 'device__model': 'A...",{'telephony_info__network_operator_name': 'SMA...,"{'accelerometer_rotation': 0, 'airplane_mode_o...","{'capability_mms': 0, 'capability_congested': ...","{'data_type': 'UidHealthStats', 'measurement_k...",2025-06-26,2288423,73d99787-c8d3-4efc-b875-80c9a9084ee1,6d776152-1d52-42c9-9dd3-bef7df31ed0e,2025-06-26 01:35:11,2025-11-17 10:26:17,2025-11-16


###  <span style='background :yellow' > Flatten the data </span>

In [7]:
dict_cols = ['GeneralInfo', 'Hardware','GeneralData','Network', 'PhoneSettings','HealthStats']

for col in dict_cols:
    print(col)
    expanded = pd.json_normalize(df_andr[col].apply(lambda x: x if isinstance(x, dict) else {}))
    expanded.index = df_andr.index   # VERY IMPORTANT
    expanded = expanded.add_prefix(f'{col}_')
    df_andr = df_andr.drop(columns=[col]).join(expanded)

GeneralInfo
Hardware
GeneralData
Network
PhoneSettings
HealthStats


In [8]:
df_andr.shape

(165989, 310)

# Device mapping 

In [9]:
df_device = pd.read_csv(r'D:\Officedata\Feature_Universe_Training\Support_Data\mobile_device_with_gsm.csv')

In [10]:
df_device.head(2)

,device_model,gsm_url,device_name,GSM_insights,run_date,Price_raw,Released_raw,device_brand,price_date
0,A100,https://www.gsmarena.com/_red_hydrogen_one-939...,RED Hydrogen One,Technology: GSM / CDMA / HSPA / EVDO / LTE; 2G...,23-09-2025,About 1200 EUR,"2018, May. Released 2018, November",Blackview,01-05-2018
1,"iPad11,7",https://www.gsmarena.com/apple_ipad_10_2_(2020...,AppleiPad 10.2 (2020),Technology: GSM / HSPA / LTE; 2G bands: GSM 85...,23-09-2025,About 370 EUR,"2020, September 15",iPad,15-09-2020


In [11]:
# normalize both sides
df_andr['model_l'] = df_andr['GeneralInfo_model'].str.lower().str.strip()
df_device['model_l'] = df_device['device_model'].str.lower().str.strip()

# create mapping dict (device_model → device_name)
model_to_name = (
    df_device
    .drop_duplicates('model_l')   # safety
    .set_index('model_l')['device_name']
)

# map
df_andr['device_name'] = df_andr['model_l'].map(model_to_name)


# 1) General Info

### mobile_brand

In [12]:
df_andr['mobile_brand'] = df_andr['GeneralInfo_brand'].str.lower().str.strip()
df_andr['mobile_brand'] = df_andr['mobile_brand'].replace('nan', np.nan)

In [13]:
# df_andr.groupby('mobile_brand', as_index=False).agg(reference_count=('credolabRefNumber', 'count'),
#         badrate=('ln_fspd30_flag', 'mean'))

In [14]:


brand_unique_list = df_andr['mobile_brand'].value_counts().loc[lambda x: x >= 20].index.tolist()



df_andr['mobile_brand_grouped'] = np.where(df_andr['mobile_brand'].isna() | (df_andr['mobile_brand'] == 'nan'), 'missing',np.where(
                                           df_andr['mobile_brand'].isin(brand_unique_list),
                                           df_andr['mobile_brand'],'low_frequency_category'))
brand_unique_list

['samsung',
 'oppo',
 'vivo',
 'realme',
 'infinix',
 'redmi',
 'honor',
 'tecno',
 'poco',
 'xiaomi',
 'huawei',
 'itel',
 'nubia',
 'google',
 'oneplus',
 'motorola',
 'zte',
 'asus',
 'nothing',
 'lge',
 'nokia',
 'lenovo',
 'wiko',
 'tcl',
 'blackshark',
 'sony']

In [15]:
# device

In [16]:
# df_andr['device_name_l'] = df_andr['device_name'].str.lower().str.strip()

In [17]:
# premium_keywords = ['ultra', 'fold', 'flip']

# high_end_brands = ['samsung']

# def is_premium_flag(brand, model):
#     if pd.isna(model):
#         return 0
    
#     brand = str(brand).lower()
#     model = str(model).lower()
    
#     # Only assign premium if brand is high-end
#     if brand in high_end_brands:
#         if any(k in model for k in premium_keywords):
#             return 1
#     return 0

# df_andr['premium_flag'] = df_andr.apply(
#     lambda x: is_premium_flag(x['mobile_brand'], x['device_name_l']),
#     axis=1
# )

In [18]:
# df_andr['premium_flag'].value_counts()

In [19]:
# df_andr[['device_name_l', 'premium_flag']].drop_duplicates()

In [20]:
# df_device.head(2)

### os version

In [21]:
df_andr['GeneralInfo_release'].unique()

<ArrowStringArray>
[   '12',    '14',    '15',    '13',    '11',     '9',    '10',    '16',
 '8.1.0', '8.0.0',   '7.0', '7.1.2',    '17', '7.1.1']
Length: 14, dtype: str

In [22]:
df_andr['os_release_version'] = df_andr['GeneralInfo_release'].str.split('.').str[0].astype(int)

In [23]:
## Create bucket
def bucket_os(version):
    if version <= 8:
        return 'old'
    elif 9 <= version <= 12:
        return 'medium'
    else:
        return 'new'

df_andr['os_version_bucket'] = df_andr['os_release_version'].apply(bucket_os)

In [24]:
df_andr['os_release_version'] = df_andr['GeneralInfo_release'].str.split('.').str[0].astype(int)


In [25]:
df_andr['device_version_age'] = df_andr['os_release_version'].max() - df_andr['os_release_version']

In [26]:
df_andr['device_version_age'].unique()

array([ 5,  3,  2,  4,  6,  8,  7,  1,  9, 10,  0])

# 2) Hardware

In [27]:
df_andr['Hardware_device__release'].unique()

<ArrowStringArray>
[   '12',    '14',    '15',    '13',    '11',     '9',    '10',    '16',
 '8.1.0', '8.0.0',   '7.0', '7.1.2',    '17', '7.1.1']
Length: 14, dtype: str

### display physical size

In [28]:
import numpy as np

df_andr['display_size_bucket'] = np.where(
    df_andr['Hardware_display__physical_size'] < 6.0, 'small_phones',
    np.where(
        df_andr['Hardware_display__physical_size'] < 6.7, 'medium_phones',
        np.where(
            df_andr['Hardware_display__physical_size'] < 7.5, 'large_phones',
            'tablet'
        )
    )
)

### Total RAM

In [29]:
df_andr['ram_in_gb'] = df_andr['Hardware_ram__ram_total_size'] / 1024

import numpy as np

df_andr['ram_bucket'] = np.where(
    df_andr['ram_in_gb'] < 4, 'low',
    np.where(
        df_andr['ram_in_gb'] < 6, 'mid',
        np.where(
            df_andr['ram_in_gb'] < 8, 'high',
            'premium'
        )
    )
)

In [30]:
df_andr['ram_bucket'].unique()

<ArrowStringArray>
['high', 'premium', 'low', 'mid']
Length: 4, dtype: str

### total main storage

In [31]:
df_andr['main_storage_gb'] = df_andr['Hardware_storage__main_storage_total'] / 1024

df_andr['main_storage_bucket'] = np.where(
    df_andr['main_storage_gb'] < 64, 'low',
    np.where(
        df_andr['main_storage_gb'] < 128, 'medium',
        np.where(
            df_andr['main_storage_gb'] < 256, 'high',
            'premium'
        )
    )
)

In [32]:
df_andr['is_high_main_storage'] = np.where(
    df_andr['main_storage_gb'] >= 128,
    1,
    0
)

### Free storage

In [33]:
df_andr['main_free_storage_gb'] = df_andr['Hardware_storage__main_storage_free'] / 1024

df_andr['storage_utilization'] =  np.where(
    df_andr['main_storage_gb'] > 0,
    (df_andr['main_storage_gb'] - df_andr['main_free_storage_gb']) / df_andr['main_storage_gb'],
    np.nan
)

### External storage

In [34]:
df_andr['ext_total_gb'] = df_andr['Hardware_storage__external_storage_total'] / 1024
df_andr['ext_free_gb'] = df_andr['Hardware_storage__external_storage_free'] / 1024

In [35]:
df_andr['has_sd_card'] = np.where(
    df_andr['Hardware_storage__external_storage_total'].notna(),
    1,
    0
)

In [36]:
df_andr['ext_storage_bucket'] = np.where(
    df_andr['ext_total_gb'].isna(), 'no_sd',
    np.where(
        df_andr['ext_total_gb'] < 32, 'small_sd',
        np.where(
            df_andr['ext_total_gb'] < 128, 'medium_sd',
            'large_sd'
        )
    )
)

In [37]:
df_andr['large_sd_flag'] = np.where(
    df_andr['ext_total_gb'] >= 128,
    1,
    0
)

### display size weight & Height

In [38]:
df_andr['display_resolution'] = np.where(
    df_andr['Hardware_display__display_size_weight'] <= 720, 'HD',
    np.where(
        df_andr['Hardware_display__display_size_weight'] <= 1080, 'Full HD',
        'QHD'
    )
)

In [39]:
import numpy as np

# First, create the height-based category
df_andr['device_display_category'] = np.where(
    df_andr['Hardware_display__display_size_height'] <= 1600, 'Low-end',
    np.where(
        (df_andr['Hardware_display__display_size_height'] > 1600) & 
        (df_andr['Hardware_display__display_size_height'] <= 2000), 'FHD+ (mid-range)',
        np.where(
            (df_andr['Hardware_display__display_size_height'] > 2000) & 
            (df_andr['Hardware_display__display_size_height'] <= 2400), 'FHD+',
            np.where(
                (df_andr['Hardware_display__display_size_height'] > 2400) & 
                (df_andr['Hardware_display__display_size_height'] <= 2800), 'Tall screens',
                'Flagship'
            )
        )
    )
)

In [40]:
df_andr['device_display_category'].unique()

<ArrowStringArray>
['FHD+ (mid-range)', 'Tall screens', 'FHD+', 'Low-end', 'Flagship']
Length: 5, dtype: str

### Sim country

In [41]:
df_andr['sim_country'] = df_andr['GeneralData_telephony_info__sim_country_iso'].fillna('unknown')

primary_country = 'ph'  # your target market / country
df_andr['is_foreign_sim'] = np.where(df_andr['sim_country'] == primary_country, 0, 1)

### locale_iso_3_country

In [42]:
df_andr['is_foreign_locale'] = np.where(
    df_andr['GeneralData_telephony_configuration__locale_iso_3_country'] == 'PHL',
    0,
    1
)

In [43]:
df_andr['locale_iso_3_country_grouped'] = np.where(df_andr['GeneralData_telephony_configuration__locale_iso_3_country'].isna() | (df_andr['mobile_brand'] == 'nan'), 'missing',np.where(
                                           df_andr['GeneralData_telephony_configuration__locale_iso_3_country'].isin(['PHL', 'USA', 'GBR']),
                                           df_andr['GeneralData_telephony_configuration__locale_iso_3_country'],'low_frequency_category'))

### signal_strength

In [44]:
import numpy as np

df_andr['signal_strength_bucket'] = np.select(
    [
        df_andr['GeneralData_signal_strength'] <= 1,
        df_andr['GeneralData_signal_strength'].between(2, 3),
        df_andr['GeneralData_signal_strength'] >= 4
    ],
    [
        'low_signal',
        'medium_signal',
        'good_signal'
    ],
    default='missing'
)

### batteryextravoltage

In [45]:
# We use 5500mV as the "Great Divide" between phones and tablets
df_andr['is_tablet'] = (df_andr['Hardware_battery__extra_voltage'] > 5500).astype(int)


# This levels the playing field so a full phone and full tablet both = 1.0
# Smartphone range: 3200-4400 | Tablet range: 6400-8800
df_andr['battery_level_pct'] = np.where(
    df_andr['is_tablet'] == 1,
    (df_andr['Hardware_battery__extra_voltage'] - 6400) / (8800 - 6400), # Tablet normalization
    (df_andr['Hardware_battery__extra_voltage'] - 3200) / (4400 - 3200)  # Phone normalization
)

# Clip to 0-1 to remove noise/errors and handle 0 voltage entries
df_andr['battery_level_pct'] = df_andr['battery_level_pct'].clip(0, 1)

# 3. Hardware Stress/Health Flag
# If voltage is below typical operating minimums (but not 0), it's a "Stress" signal
df_andr['hardware_stress_signal'] = (
    ((df_andr['is_tablet'] == 0) & (df_andr['Hardware_battery__extra_voltage'] < 3400) & (df_andr['Hardware_battery__extra_voltage'] > 0)) |
    ((df_andr['is_tablet'] == 1) & (df_andr['Hardware_battery__extra_voltage'] < 6600) & (df_andr['Hardware_battery__extra_voltage'] > 0))
).astype(int)




### batteryextraplugged

In [46]:
import numpy as np

battery_map = {
    0: 'on_battery',
    1: 'ac_charger',
    2: 'usb_charger',
    4: 'wireless_charger'
}

df_andr['battery_plugged_type'] = (
    df_andr['Hardware_battery__extra_plugged']
    .map(battery_map)
    .fillna('missing')
)

### battery_stress_flag / battery_temp_bin

In [47]:
# Android reports temperature in tenths of °C.
df_andr['battery_temp_c'] = df_andr['Hardware_battery__extra_temperature'] / 10

df_andr['battery_stress_flag'] = np.where(
    df_andr['battery_temp_c'] > 40,
    1,
    0
)

In [48]:

df_andr['battery_temp_bin'] = np.select(
    [
        df_andr['battery_temp_c'] < 0,                     # negative / invalid
        df_andr['battery_temp_c'] < 30,
        df_andr['battery_temp_c'].between(30, 35),
        df_andr['battery_temp_c'].between(35, 40),
        df_andr['battery_temp_c'] > 40
    ],
    [
        'negative_invalid',
        'cool',
        'normal',
        'warm',
        'hot'
    ],
    default='missing'
)

### Hardware_cpu__core_count

In [49]:
# Categorizing the cores into risk-relevant segments
conditions = [
    (df_andr['Hardware_cpu__core_count'] >= 9),        # High-End/Specialist
    (df_andr['Hardware_cpu__core_count'] == 8),        # Modern Standard
    (df_andr['Hardware_cpu__core_count'] == 6),        # Mid-Tier
    (df_andr['Hardware_cpu__core_count'] <= 4),        # Budget/Legacy
    (df_andr['Hardware_cpu__core_count'].isna())       # Missing/Unknown
]
choices = ['Premium', 'Standard','Mid-Tier', 'Budget', 'Unknown/Emulator']
df_andr['cpu_core_bin'] = np.select(conditions, choices, default='Tier_Unknown')

### Hardware_cpu__cpu_abi

In [50]:
df_andr['Hardware_cpu__cpu_abi'].value_counts(dropna=False)

Hardware_cpu__cpu_abi
arm64-v8a;armeabi-v7a;armeabi    141858
arm64-v8a                         12650
armeabi-v7a;armeabi                6362
NaN                                5062
armeabi-v7a                          57
Name: count, dtype: int64

In [51]:
df_andr['Hardware_cpu__cpu_abi'].unique()

<ArrowStringArray>
['arm64-v8a;armeabi-v7a;armeabi',           'armeabi-v7a;armeabi',
                     'arm64-v8a',                             nan,
                   'armeabi-v7a']
Length: 5, dtype: str

In [52]:
df_andr['cpu_arch_bin'] = np.select(
    [
           df_andr['Hardware_cpu__cpu_abi'].str.contains('arm64', na=False),
        df_andr['Hardware_cpu__cpu_abi'].str.contains('armeabi-v7a', na=False),
        df_andr['Hardware_cpu__cpu_abi'].isna()
    ],
    [
        'modern_64bit',
        'old_32bit',
        'missing'
    ],
    default='other'   
    )

### wifi_speed_bin

In [53]:
import numpy as np

df_andr['wifi_speed_bin'] = np.select(
    [
        df_andr['Hardware_wifi__connection_info_link_speed'] == -1,
        df_andr['Hardware_wifi__connection_info_link_speed'] < 50,
        df_andr['Hardware_wifi__connection_info_link_speed'].between(50, 150),
        df_andr['Hardware_wifi__connection_info_link_speed'].between(151, 500),
        df_andr['Hardware_wifi__connection_info_link_speed'] > 500
    ],
    [
        'no_wifi_or_unknown',
        'very_low_speed',
        'medium_speed',
        'high_speed',
        'very_high_speed'
    ],
    default='missing'
)

In [54]:
import pandas as pd
import numpy as np

# Updated to use df_andr
# Note: Ensure IP is treated as a string
df_andr['dhcp_ip'] = df_andr['Hardware_wifi__dhcp_info_server_address'].astype(str)

# Network Stability Binning
def bin_network_type(ip):
    if ip in ['192.168.1.1', '192.168.0.1', '192.168.2.1']:
        return 'Standard_Home'
    elif ip.startswith('192.168.43.') or ip.startswith('172.20.'):
        return 'Mobile_Hotspot'
    elif ip.startswith('10.'):
        return 'Enterprise_Corporate'
    elif ip == '0.0.0.0':
        return 'Disconnected'
    else:
        return 'Custom_Config'

df_andr['dhcp_network_tier'] = df_andr['dhcp_ip'].apply(bin_network_type)


### Hardware_common__phone_uptime_ms

In [55]:
# Convert ms to days
df_andr['phone_uptime_days'] = df_andr['Hardware_common__phone_uptime_ms'] / 86_400_000


def uptime_bin(days):
    if days < 1:
        return 'Very_Short'
    elif days < 7:
        return 'Short'
    elif days < 30:
        return 'Medium'
    else:
        return 'Long'

df_andr['phone_uptime_category'] = df_andr['phone_uptime_days'].apply(uptime_bin)

### Hardware_common__is_fingerprint_hardware_detected

# 3) Network

In [56]:
[col for col in df_andr.columns if col.startswith('Network')]

['Network_capability_mms',
 'Network_capability_congested',
 'Network_capability_not_metered',
 'Network_capability_not_restricted',
 'Network_capability_not_roaming',
 'Network_capability_not_suspended',
 'Network_capability_rcs',
 'Network_capability_supl',
 'Network_capability_temporarily_not_metered',
 'Network_capability_trusted',
 'Network_capability_validated',
 'Network_capability_wifi_p_2_p',
 'Network_capability_xcap',
 'Network_transport_bluetooth',
 'Network_transport_cellular',
 'Network_transport_ethernet',
 'Network_transport_lowpan',
 'Network_transport_vpn',
 'Network_transport_wifi',
 'Network_link_downstream_bandwidth_kbps',
 'Network_link_upstream_bandwidth_kbps',
 'Network_signal_strength']

### downstream_bandwidth_cat

In [57]:
df_andr['downstream_mbps'] = df_andr['Network_link_downstream_bandwidth_kbps'] / 1000

df_andr['downstream_mbps_clean'] = df_andr['downstream_mbps'].apply(
    lambda x: x if x < 1000 else None
)

def bandwidth_category(mbps):
    if mbps is None:
        return 'Unknown'
    elif mbps < 5:
        return 'Low_bandwidth'
    elif mbps < 50:
        return 'Medium_bandwidth'
    else:
        return 'High_bandwidth'

df_andr['downstream_bandwidth_cat'] = df_andr['downstream_mbps_clean'].apply(bandwidth_category)

### upstream_bandwidth_cat

In [58]:
df_andr['upstream_mbps'] = df_andr['Network_link_upstream_bandwidth_kbps'] / 1000

df_andr['upstream_mbps_clean'] = df_andr['upstream_mbps'].apply(
    lambda x: x if x < 1000 else None
)

def upstream_bandwidth_category(mbps):
    if mbps is None:
        return 'Unknown'
    elif mbps < 2:
        return 'Low_bandwidth'
    elif mbps < 20:
        return 'Medium_bandwidth'
    else:
        return 'High_bandwidth'

df_andr['upstream_bandwidth_cat'] = df_andr['upstream_mbps_clean'].apply(upstream_bandwidth_category)

### signal_strength_cat

In [59]:
def signal_strength_category(signal):
    if pd.isna(signal) or signal in [0, 99]:
        return 'Unknown'
    elif signal <= 30:
        return 'Very_Weak'
    elif signal <= 50:
        return 'Weak'
    elif signal <= 70:
        return 'Medium'
    elif signal <= 85:
        return 'Strong'
    else:
        return 'Very_Strong'

df_andr['signal_strength_cat'] = df_andr['Network_signal_strength'].apply(signal_strength_category)

# 4) PhoneSettings

In [60]:
[col for col in df_andr.columns if col.startswith('PhoneSettings')]

['PhoneSettings_accelerometer_rotation',
 'PhoneSettings_airplane_mode_on',
 'PhoneSettings_always_finish_activities',
 'PhoneSettings_animator_duration_scale',
 'PhoneSettings_auto_time',
 'PhoneSettings_auto_time_zone',
 'PhoneSettings_bluetooth_on',
 'PhoneSettings_data_roaming',
 'PhoneSettings_device_provisioned',
 'PhoneSettings_install_non_market_apps',
 'PhoneSettings_mode_ringer',
 'PhoneSettings_transition_animation_scale',
 'PhoneSettings_usb_mass_storage_enabled',
 'PhoneSettings_wifi_networks_available_notification_on',
 'PhoneSettings_wifi_watchdog_on',
 'PhoneSettings_adb_enabled',
 'PhoneSettings_accessibility_enabled',
 'PhoneSettings_development_settings_enabled',
 'PhoneSettings_wifi_max_dhcp_retry_count',
 'PhoneSettings_boot_count',
 'PhoneSettings_debug_app',
 'PhoneSettings_wifi_mobile_data_transition_wakelock_timeout_ms',
 'PhoneSettings_wifi_networks_available_repeat_delay',
 'PhoneSettings_wifi_num_ope_networks_kept',
 'PhoneSettings_contact_metadata_sync_enab

In [61]:
def ringing_mode_cat(x):
    if x == 0:
        return 'Silent'
    elif x == 1:
        return 'Vibrate'
    elif x == 2:
        return 'Normal'
    else:
        return 'Unknown'

df_andr['ringing_mode_cat'] = df_andr['PhoneSettings_ringing_mode'].apply(ringing_mode_cat)

# df_andr.groupby('ringing_mode_cat', dropna=False, as_index=False).agg(reference_count=('digitalLoanAccountId', 'count'),    badrate=('ln_fspd30_flag', 'mean'))

In [62]:
# Convert milliseconds → seconds
df_andr['screen_off_timeout_sec'] = df_andr['PhoneSettings_screen_off_timeout'] / 1000

def screen_off_timeout_cat_sec(sec):
    if pd.isna(sec) or sec > 2e6 or sec == 0:
        return 'Unknown'
    elif sec <= 30:
        return 'Very_Short'
    elif sec <= 120:
        return 'Short'
    elif sec <= 600:
        return 'Medium'
    elif sec <= 1800:
        return 'Long'
    else:
        return 'Unknown'

df_andr['screen_off_timeout_cat_sec'] = df_andr['screen_off_timeout_sec'].apply(screen_off_timeout_cat_sec)

In [63]:
def wifi_notification_risk(val):
    if pd.isna(val) or val == 0:
        return 1  # high-risk / not notified
    else:
        return 0  # normal / lower risk

df_andr['wifi_notify_risk_flag'] = df_andr['PhoneSettings_wifi_networks_available_notification_on'].apply(wifi_notification_risk)

# df_andr.groupby('wifi_notify_risk_flag', dropna=False, as_index=False).agg(reference_count=('digitalLoanAccountId', 'count'),    badrate=('ln_fspd30_flag', 'mean'))

In [64]:
def animation_scale_cat(scale):
    if pd.isna(scale):
        return 'Unknown'
    elif scale == 0.0:
        return 'Off'
    elif scale < 1.0:
        return 'Reduced'
    elif scale == 1.0:
        return 'Normal'
    else:  # >1.0
        return 'High'

df_andr['animation_scale_cat'] = df_andr['PhoneSettings_transition_animation_scale'].apply(animation_scale_cat)

# df_andr.groupby('animation_scale_cat', dropna=False, as_index=False).agg(reference_count=('digitalLoanAccountId', 'count'),    badrate=('ln_fspd30_flag', 'mean')).sort_values(
#     by='reference_count', 
#     ascending=False
# )

In [65]:
def ringer_risk_mode(mode):
    if mode == 0:
        return 'Silent'
    elif mode == 1:
        return 'Vibrate'
    elif mode == 2:
        return 'Normal'
    else:
        return 'Unknown'

df_andr['ringer_mode_cat'] = df_andr['PhoneSettings_mode_ringer'].apply(ringer_risk_mode)

In [66]:
# df_andr.groupby('ringer_mode_cat', dropna=False, as_index=False).agg(reference_count=('digitalLoanAccountId', 'count'),    badrate=('ln_fspd30_flag', 'mean')).sort_values(
#     by='reference_count', 
#     ascending=False
# )

# 5) GeneralData

In [67]:
[col for col in df_andr.columns if col.startswith('GeneralData')]

['GeneralData_telephony_info__network_operator_name',
 'GeneralData_telephony_info__sim_country_iso',
 'GeneralData_telephony_info__network_operator',
 'GeneralData_telephony_info__phone_type',
 'GeneralData_telephony_info__hashed_android_id',
 'GeneralData_telephony_configuration__mcc',
 'GeneralData_telephony_configuration__mnc',
 'GeneralData_telephony_configuration__keyboard',
 'GeneralData_telephony_configuration__locale_iso_3_language',
 'GeneralData_telephony_configuration__locale_iso_3_country',
 'GeneralData_telephony_configuration__time_zone_id',
 'GeneralData_telephony_configuration__locale_display_language',
 'GeneralData_telephony_info__sim_serial_number',
 'GeneralData_telephony_info__phone_number',
 'GeneralData_telephony_info__device_software_version',
 'GeneralData_telephony_info__sim_operator',
 'GeneralData_telephony_info__device_id',
 'GeneralData_telephony_info__android_id',
 'GeneralData_telephony_configuration__ui_mode',
 'GeneralData_telephony_configuration__tou

In [68]:
mcc_country_map = {
    0: "Unknown",
    202: "Greece",
    204: "Netherlands",
    214: "Spain",
    240: "Sweden",
    260: "Poland",
    278: "Malta",
    302: "Canada",
    310: "United States",
    419: "Kuwait/Region 4xx",
    420: "Saudi Arabia",
    422: "Oman",
    424: "United Arab Emirates",
    454: "Hong Kong",
    460: "China",
    466: "Taiwan",
    472: "Maldives",
    502: "Malaysia",
    505: "Australia",
    510: "Indonesia",
    515: "Philippines",
    520: "Thailand",
    525: "Singapore",
    530: "New Zealand",
    541: "Vanuatu",
    999: "Reserved/Unknown"
}

df_andr['network_country'] = df_andr['GeneralData_telephony_configuration__mcc'].map(mcc_country_map)

# df_andr.groupby('network_country', dropna=False, as_index=False).agg(reference_count=('digitalLoanAccountId', 'count'),    badrate=('ln_fspd30_flag', 'mean'))

# 6) Healthstats

In [69]:
[col for col in df_andr.columns if col.startswith('Health')]

['HealthStats_data_type',
 'HealthStats_measurement_key_count',
 'HealthStats_stats_key_count',
 'HealthStats_timer_key_count',
 'HealthStats_measurement_realtime_battery_ms',
 'HealthStats_measurement_uptime_battery_ms',
 'HealthStats_measurement_realtime_screen_off_battery_ms',
 'HealthStats_measurement_uptime_screen_off_battery_ms',
 'HealthStats_measurement_wifi_idle_ms',
 'HealthStats_measurement_wifi_rx_ms',
 'HealthStats_measurement_wifi_tx_ms',
 'HealthStats_measurement_wifi_power_mams',
 'HealthStats_measurement_bluetooth_idle_ms',
 'HealthStats_measurement_bluetooth_rx_ms',
 'HealthStats_measurement_bluetooth_tx_ms',
 'HealthStats_measurement_bluetooth_power_mams',
 'HealthStats_measurement_mobile_idle_ms',
 'HealthStats_measurement_mobile_rx_ms',
 'HealthStats_measurement_mobile_tx_ms',
 'HealthStats_measurement_mobile_power_mams',
 'HealthStats_measurement_wifi_running_ms',
 'HealthStats_measurement_wifi_full_lock_ms',
 'HealthStats_measurement_wifi_multicast_ms',
 'HealthS

In [70]:
# df_andr.groupby('HealthStats_measurement_realtime_screen_off_battery_ms', dropna=False, as_index=False).agg(reference_count=('digitalLoanAccountId', 'count'),    badrate=('ln_fspd30_flag', 'mean'))

In [71]:
df_andr['HealthStats_timer_process_state_top_ms_time'].unique()

array([ 176774.,  158253.,   30390., ..., 1818844.,   19024.,  250340.],
      shape=(117641,))

In [72]:
df_andr['health_realtime_screen_off_hours'] = df_andr['HealthStats_measurement_realtime_screen_off_battery_ms'] / (1000*60*60)

In [73]:
df_andr['health_realtime_screen_off_hours'].nunique()

154847

### foreground_time

In [74]:
df_andr['foreground_time_per_use'] = (
    df_andr['HealthStats_timer_foreground_activity_time'] / 
    df_andr['HealthStats_timer_foreground_activity_count'].replace(0, np.nan)
)

df_andr['foreground_intensity'] = (
    df_andr['HealthStats_timer_foreground_activity_time'] * df_andr['HealthStats_timer_foreground_activity_count']
)

### camera time 

In [75]:
df_andr['camera_time_per_use'] = (
    df_andr['HealthStats_timer_camera_time'] / 
    df_andr['HealthStats_timer_camera_count'].replace(0, np.nan)
)

df_andr['camera_intensity'] = (
    df_andr['HealthStats_timer_camera_time'] * df_andr['HealthStats_timer_camera_count']
)

### timer_wifi_scan

In [76]:
df_andr['wifi_scan_intensity'] = (
    df_andr['HealthStats_timer_wifi_scan_time'] * df_andr['HealthStats_timer_wifi_scan_count']
)

df_andr['wifi_scan_time_per_scan'] = (
    df_andr['HealthStats_timer_wifi_scan_time'] / 
    df_andr['HealthStats_timer_wifi_scan_count'].replace(0, np.nan)
)

### system_user_cpu_ratio

In [77]:
df_andr['system_cpu_sec'] = df_andr['HealthStats_measurement_system_cpu_time_ms'] / 1000
df_andr['user_cpu_sec'] = df_andr['HealthStats_measurement_user_cpu_time_ms'] / 1000

df_andr['system_user_cpu_ratio'] = (
    df_andr['system_cpu_sec'] / df_andr['user_cpu_sec'].replace(0, np.nan)
)

### wifi_rx_tx_packet_ratio

In [78]:
df_andr['wifi_rx_tx_packet_ratio'] = (
    df_andr['HealthStats_measurement_wifi_rx_packets'] /
    df_andr['HealthStats_measurement_wifi_tx_packets'].replace(0, np.nan)
)

# >1 → mostly downloads
# <1 → mostly uploads

### mobile_rx_tx_packet_ratio

In [79]:
df_andr['mobile_rx_tx_packet_ratio'] = (
    df_andr['HealthStats_measurement_mobile_rx_packets'] /
    df_andr['HealthStats_measurement_mobile_tx_packets'].replace(0, np.nan)
)

# >1 → mostly downloads
# <1 → mostly uploads

### wifi_rx_tx_ratio

In [80]:

df_andr['wifi_rx_tx_ratio'] = (
    df_andr['HealthStats_measurement_wifi_rx_bytes'] /
    df_andr['HealthStats_measurement_wifi_tx_bytes'].replace(0, np.nan)
)
# Ratio	Behavior
#>1 → mostly downloads
# <1 → mostly uploads

### mobile_download_upload_ratio

In [81]:

df_andr['mobile_download_upload_ratio'] = (
    df_andr['HealthStats_measurement_mobile_rx_bytes'] /
    df_andr['HealthStats_measurement_mobile_tx_bytes'].replace(0, np.nan)
)
# Ratio	Behavior
# >5	content consumption
# ~1	balanced usage
# <1	heavy uploads


### wifi_full_lock_hours

In [82]:
df_andr['wifi_full_lock_hours'] = (
    df_andr['HealthStats_measurement_wifi_full_lock_ms'] / 3600000
)

### mobile rx tx

In [83]:
df_andr['mobile_total_activity'] = (
    df_andr['HealthStats_measurement_mobile_rx_ms'].fillna(0) +
    df_andr['HealthStats_measurement_mobile_tx_ms'].fillna(0)
)
# Overall mobile network usage.


df_andr['mobile_rx_tx_ratio'] = (
    df_andr['HealthStats_measurement_mobile_rx_ms'] /
    df_andr['HealthStats_measurement_mobile_tx_ms'].replace(0, np.nan)
)
# Ratio	Meaning
# >1	Mostly downloading
# <1	More uploading
# Very high	Passive consumption



df_andr['mobile_upload_share'] = (
    df_andr['HealthStats_measurement_mobile_tx_ms'] /
    df_andr['mobile_total_activity'].replace(0, np.nan)
)
# Share	Behavior
# Low	Content consumption
# High	Content creation / messaging

### Wifi rx, tx 

In [84]:
df_andr['wifi_total_time'] = (
    df_andr['HealthStats_measurement_wifi_idle_ms'].fillna(0) +
    df_andr['HealthStats_measurement_wifi_rx_ms'].fillna(0) +
    df_andr['HealthStats_measurement_wifi_tx_ms'].fillna(0)
)
# Total time Wi-Fi interface was active.



df_andr['wifi_active_time'] = (
    df_andr['HealthStats_measurement_wifi_rx_ms'].fillna(0) +
    df_andr['HealthStats_measurement_wifi_tx_ms'].fillna(0)
)
# Value	Meaning
# High	Heavy internet usage
# Low	Idle device




df_andr['wifi_idle_ratio'] = (
    df_andr['HealthStats_measurement_wifi_idle_ms'] /
    df_andr['wifi_total_time'].replace(0, np.nan)
)
# Ratio	Behavior
# ~0.1–0.3	Active Wi-Fi usage
# ~0.5–0.8	Mostly idle
# ~1	Connected but unused


df_andr['wifi_rx_tx_ratio'] = (
    df_andr['HealthStats_measurement_wifi_rx_ms'] /
    df_andr['HealthStats_measurement_wifi_tx_ms'].replace(0, np.nan)
)
# >1	Mostly downloading
# <1	Mostly uploading

df_andr['wifi_activity_score'] = np.log1p(
    df_andr['wifi_active_time']
)

### screen_off_cpu_activity_ratio

In [85]:
df_andr['screen_off_cpu_activity_ratio'] = (
    df_andr['HealthStats_measurement_uptime_screen_off_battery_ms'] /
    df_andr['HealthStats_measurement_realtime_screen_off_battery_ms']
)

### screen_off_ratio

In [86]:
df_andr['screen_off_ratio'] = (
    df_andr['HealthStats_measurement_realtime_screen_off_battery_ms'] /
    df_andr['HealthStats_measurement_realtime_battery_ms']
)

### system_user_ratio

In [87]:
df_andr['system_user_ratio'] = (
    df_andr['HealthStats_proc_measurement_system_time_ms'] /
    df_andr['HealthStats_proc_measurement_user_time_ms'].replace(0, np.nan)
)

### Other ratio fields

In [88]:
df_andr['top_avg_sec'] = df_andr['HealthStats_timer_process_state_top_ms_time'] / df_andr['HealthStats_timer_process_state_top_ms_count'].replace(0, np.nan)
df_andr['background_avg_sec'] = df_andr['HealthStats_timer_process_state_background_ms_time'] / df_andr['HealthStats_timer_process_state_background_ms_count'].replace(0, np.nan)
df_andr['cached_avg_sec'] = df_andr['HealthStats_timer_process_state_cached_ms_time'] / df_andr['HealthStats_timer_process_state_cached_ms_count'].replace(0, np.nan)
df_andr['vibrator_avg_sec'] = df_andr['HealthStats_timer_vibrator_time'] / df_andr['HealthStats_timer_vibrator_count'].replace(0, np.nan)
df_andr['mobile_radio_avg_sec'] = df_andr['HealthStats_timer_mobile_radio_active_time'] / df_andr['HealthStats_timer_mobile_radio_active_count'].replace(0, np.nan)

### HealthStats_proc_measurement_crashes_count

In [89]:
def crash_category(x):
    if pd.isna(x):
        return 'Unknown'
    elif x == 0:
        return 'Stable'
    elif x <= 2:
        return 'Moderate'
    else:
        return 'Unstable'

df_andr['proc_measurement_crash_cat'] = df_andr['HealthStats_proc_measurement_crashes_count'].apply(crash_category)
# df_andr.groupby('proc_measurement_crash_cat', dropna=False, as_index=False).agg(reference_count=('digitalLoanAccountId', 'count'),    badrate=('ln_fspd30_flag', 'mean'))

### HealthStats_proc_anr_count

In [90]:
def anr_category(x):
    if pd.isna(x):
        return 'Unknown'
    elif x == 0:
        return 'Stable'
    elif x <= 2:
        return 'Moderate'
    else:
        return 'Unstable'

df_andr['proc_anr_category'] = df_andr['HealthStats_proc_anr_count'].apply(anr_category)



In [91]:
# df_andr.groupby('proc_anr_category', dropna=False, as_index=False).agg(reference_count=('digitalLoanAccountId', 'count'),    badrate=('ln_fspd30_flag', 'mean'))

### device_risk_flag

In [92]:
df_andr['device_risk_flag'] = (
    (df_andr['GeneralData_common__is_rooted'] == 1) |
    (df_andr['GeneralData_common__is_user_demo'] == 1) |
    (df_andr['GeneralData_common__is_user_unlocked'] == 0)
).astype(int)

In [93]:
# df_andr.groupby('device_risk_flag', dropna=False, as_index=False).agg(reference_count=('digitalLoanAccountId', 'count'),    badrate=('ln_fspd30_flag', 'mean'))

### collection_duration_min

In [94]:
df_andr['collection_duration_min'] = (
    (df_andr['GeneralInfo_end_collect_time'] -
     df_andr['GeneralInfo_start_collect_time'])
    .dt.total_seconds()  * 1000
)

### network_quality

In [95]:
## log transform remoes skewness in data, bcz download bandwidth is skewed. also to scale signal strength 0-1 we are dividing it by 99
df_andr['network_quality'] = (
    np.log1p(df_andr['Network_link_downstream_bandwidth_kbps']) +
    np.log1p(df_andr['Network_link_upstream_bandwidth_kbps']) +
    (df_andr['Network_signal_strength'] / 99)
)

In [96]:
# df_andr.groupby('network_quality', dropna=False, as_index=False).agg(reference_count=('digitalLoanAccountId', 'count'),    badrate=('ln_fspd30_flag', 'mean'))
df_andr['network_quality'].unique()

array([19.51636774, 22.08308634, 19.56515054, ..., 19.49203844,
       20.59367913, 22.15432901], shape=(79223,))

### storage_free_ratio

In [97]:
df_andr['storage_free_ratio'] = (
    df_andr['Hardware_storage__main_storage_free'] /
    df_andr['Hardware_storage__main_storage_total'].replace(0, np.nan)
)

### activity_intensity

In [98]:
df_andr['activity_intensity'] = (
    np.log1p(df_andr['HealthStats_measurement_user_cpu_time_ms'].fillna(0)) +
    df_andr['HealthStats_timer_foreground_activity_count'].fillna(0)
)

### phone_uptime_hours

In [99]:
df_andr['Hardware_common__phone_uptime_ms'].unique()

array([5.15701580e+08, 2.20152099e+08, 1.03601916e+08, ...,
       2.01500078e+08, 3.94598166e+08, 1.24341900e+06], shape=(156270,))

In [100]:
df_andr['phone_uptime_hours'] = (
    df_andr['Hardware_common__phone_uptime_ms'] / 3600000
)

### device_power_score

In [101]:
df_andr['Hardware_cpu__cpu_frequency'].unique()

array([None], dtype=object)

In [102]:
df_andr['device_power_score'] = (
    df_andr['Hardware_cpu__core_count'].fillna(0) *
    df_andr['Hardware_cpu__cpu_frequency'].fillna(0)
)

### Network_usage

In [103]:
df_andr['HealthStats_measurement_mobile_tx_bytes'].unique()

array([      0.,  144669., 1302172., ...,  155450.,  223173.,    5014.],
      shape=(44725,))

In [104]:
df_andr['Network_usage'] = (
    df_andr['HealthStats_measurement_mobile_rx_bytes'] +
    df_andr['HealthStats_measurement_mobile_tx_bytes']
)

### wifi_data

In [105]:
df_andr['wifi_data'] = (
    df_andr['HealthStats_measurement_wifi_rx_packets'] +
    df_andr['HealthStats_measurement_wifi_tx_packets']
)

### entertainment_usage

In [106]:
# entertainment_usage =timervideotime+ timeraudiotime

df_andr['entertainment_usage'] = (
    df_andr['HealthStats_timer_audio_time'].fillna(0) +
    df_andr['HealthStats_timer_video_time'].fillna(0)
)

df_andr['entertainment_usage_in_hrs'] = (df_andr['entertainment_usage'] / 3_600_000).round(1)

In [107]:
np.set_printoptions(suppress=True)

In [108]:
df_andr['entertainment_usage_in_hrs'].unique()

array([  0. ,   0.5,   0.3,   3.1,   2.1,   0.8,   5.5,   0.1,   2.2,
         0.6,   3.5,   1.1,   2.4,   0.2,   6.8,   1.2,  15.6,   0.4,
         3.3,   5.2,  16. ,   1.3,  11. ,   3. ,   0.7,   4.2,   6.9,
         1.8,   5.6,   2.3,  28.1,   3.8,   1.5,   6.7,   3.7,   1.6,
         2.6,   1. ,  11.1,   1.4,  19.4,  23.9,   5.3,   5.8,  19.2,
         1.9,   8.3,   0.9,   4.4,  18.4,   5. ,  49.7,  21.4,   2.9,
         5.4,   1.7,   4.3,   7.5,   9.9,   2. ,   2.5,  18.2,   8. ,
         6.3,   3.9,  12.8,  68.6, 111.9,   7.8,   2.7,  39.1,  24.4,
        17.4,   3.4,  22.9,  11.7,  21.8,  10.1,   6.4,   3.2,   4.6,
        18. ,  12.4,   5.9,  10.8,   7.7,  13.8,   8.2,  10. ,  18.1,
        69.6,  10.9,  19. ,   7. ,  35. ,  12.2,  12.9,  14. ,  24.3,
         6.6,  20.9,  26.5,  16.2,  19.1,   9.7,   8.6,   7.3,   3.6,
        16.1,   4.1,   5.1,  17. ,  25.1,   4.8, 122.7,   9.1,   2.8,
        11.4,   8.8,   8.5,  41.9,  10.2,  32.5,   9.4,   4.7,  10.6,
       135. ,  64.2,

### data_usage

In [109]:
# Fill NaNs and calculate total usage
df_andr['data_usage'] = (
    df_andr['HealthStats_measurement_mobile_rx_bytes'] +
    df_andr['HealthStats_measurement_mobile_tx_bytes'] +
    df_andr['HealthStats_measurement_wifi_rx_packets'] +
    df_andr['HealthStats_measurement_wifi_tx_packets']
)

df_andr['data_usage_mb'] = (df_andr['data_usage'] / 1_000_000).round(2)

# Log-transform for modeling
# df_andr['log_data_usage'] = np.log1p(df_andr['data_usage'])

In [110]:
# df_andr['data_usage_mb'].unique()

In [111]:
# storagemainstoragetotal - storage
# cpucorecount - CPU Core
# cpucpufrequency - CPU power
# displayphysicalsize - touch screen size
# cameracameraresolution - camera resolution
# cameracamercount - Count of camera

### battery_health_score

In [112]:
df_andr['Hardware_battery__extra_health'].unique()

array([ 2,  1,  9,  6,  8,  4,  7, 10,  3])

In [113]:
df_andr['battery_health_scaled'] = df_andr['Hardware_battery__extra_health'] / 10
df_andr['battery_health_score'] = 1 - df_andr['battery_health_scaled']

In [114]:
df_andr['battery_health_score'].unique()

array([0.8, 0.9, 0.1, 0.4, 0.2, 0.6, 0.3, 0. , 0.7])

### security_flag

In [115]:
df_andr['security_settings_score'] = (
    df_andr['PhoneSettings_has_enrolled_fingerprints'] +
    df_andr['PhoneSettings_location_enabled'] +
    df_andr['PhoneSettings_auto_time'] +
    df_andr['PhoneSettings_auto_time_zone']
) / 4

In [116]:
df_andr['security_settings_flag'] = np.where(
                                    (df_andr['PhoneSettings_has_enrolled_fingerprints'] == 1) |
                                    (df_andr['PhoneSettings_location_enabled'] == 1) |
                                    (df_andr['PhoneSettings_auto_time'] == 1) |
                                    (df_andr['PhoneSettings_auto_time_zone'] == 1),
                                    1,  # value if True
                                    0   # value if False
                                )

### vpn_flag

In [117]:
df_andr['vpn_flag'] = df_andr['Network_transport_vpn']

In [118]:
df_andr['Network_transport_vpn'].unique()

array([ 0.,  1., nan])

In [119]:
df_andr['Hardware_cpu__core_count'].unique()

array([ 8.,  4., 10.,  9., nan,  6.])

In [120]:
df_andr['PhoneSettings_development_settings_enabled'].unique()

array([nan,  0.,  1.])

In [121]:
df_andr['tech_savvy_flag'] = np.where(
    (df_andr['PhoneSettings_install_non_market_apps'] == 1) |
    (df_andr['PhoneSettings_development_settings_enabled'] == 1) |
    (df_andr['PhoneSettings_adb_enabled'] == 1),
    1,  # tech-savvy / potential higher fraud risk
    np.where(
        (df_andr['PhoneSettings_install_non_market_apps'].isna()) &
        (df_andr['PhoneSettings_development_settings_enabled'].isna()) &
        (df_andr['PhoneSettings_adb_enabled'].isna()),
        np.nan,  # keep as NaN if all are missing
        0  # else 0 (none enabled)
    )
)

In [122]:
# df_andr['tech_savvy_score'].unique()

In [123]:
df_andr['tech_savvy_score'] = (
    df_andr[['PhoneSettings_install_non_market_apps',
             'PhoneSettings_development_settings_enabled',
             'PhoneSettings_adb_enabled']].sum(axis=1, min_count=1)
)

### font_scale_bin

In [124]:
df_andr['font_scale_bin'] = np.where(
    (df_andr['PhoneSettings_font_scale'] < 0.9) | (df_andr['PhoneSettings_font_scale'].isna()),
    '<0.9_or_blank',
    np.where(
        (df_andr['PhoneSettings_font_scale'] >= 0.9) & (df_andr['PhoneSettings_font_scale'] <= 1),
        '0.9_to_1',
        '>1'
    )
)

In [125]:
# df_andr.groupby('font_scale_bin', dropna=False, as_index=False).agg(reference_count=('digitalLoanAccountId', 'count'),    badrate=('ln_fspd30_flag', 'mean'))

### battery_level_bin

In [126]:
df_andr['Hardware_battery__extra_level'].unique()

array([ 30,  57,  74,   8,  90,  41,  80,   9,  62,   4,  63,  43,  47,
        25,  88,  67,  96,  60,  77,  94, 100,  11,  55,  79,  87,  99,
        49,  37,  31,  33,  22,  35,  12,  20,  48,  17,  51,  26,  59,
        52,  44,  65,  61,  13,  86,  78,  53,  72,  18,  40,  39,  42,
        29,  76,  36,  82,  46,  71,  93,  81,  85,  32,  21,  91,  10,
        69,  89,  95,  92,  68,  38,  28,  75,  70,  98,  58,  84,  16,
        45,  24,  23,  50,  54,  83,  56,   6,  64,  14,  73,  19,  97,
        66,  34,  27,   2,  15,   7,   5,   1,   3,   0])

In [127]:
bins = [0,10,20,30,40,50,60,70,80,90,100]

labels = [
    '0-9',
    '10-19',
    '20-29',
    '30-39',
    '40-49',
    '50-59',
    '60-69',
    '70-79',
    '80-89',
    '90-100'
]

df_andr['battery_level_bin'] = pd.cut(
    df_andr['Hardware_battery__extra_level'],
    bins=bins,
    labels=labels,
    right=False
)
df_andr['battery_level_bin'] =df_andr['battery_level_bin'].astype('object')

In [128]:
# df_andr.groupby('battery_level_bin', dropna=False, as_index=False).agg(reference_count=('digitalLoanAccountId', 'count'),    badrate=('ln_fspd30_flag', 'mean'))

### router_type

In [129]:
def router_category(ip):
    
    if pd.isna(ip) or ip == '0.0.0.0':
        return 'No_wifi'
    
    elif ip.startswith('192.168'):
        return 'Home_router'
    
    elif ip.startswith('10.'):
        return 'Corporate_network'
    
    elif ip.startswith('172.'):
        second = int(ip.split('.')[1])
        if 16 <= second <= 31:
            return 'Corporate_network'
        else:
            return 'Public_network'
    
    else:
        return 'Public_network'
    
df_andr['router_type'] = df_andr['Hardware_wifi__dhcp_info_server_address'].apply(router_category)
df_andr['router_type'].value_counts()

router_type
Home_router          109141
No_wifi               47395
Corporate_network      9012
Public_network          441
Name: count, dtype: int64

In [130]:
feature_list = [
    'customerid',
    'Observation_date',
    # "mobile_brand_grouped",
    "os_version_bucket",
    "device_version_age",
    "display_size_bucket",
    "ram_bucket",
    "main_storage_bucket",
    "is_high_main_storage",
    "storage_utilization",
    "has_sd_card",
    "ext_storage_bucket",
    "large_sd_flag",
    "display_resolution",
    "device_display_category",
    "is_foreign_sim",
    "is_foreign_locale",
    "locale_iso_3_country_grouped",
    "signal_strength_bucket",
    "is_tablet",
    "battery_level_pct",
    "hardware_stress_signal",
    "battery_plugged_type",
    "battery_stress_flag",
    "battery_temp_bin",
    "cpu_core_bin",
    "cpu_arch_bin",
    "wifi_speed_bin",
    "phone_uptime_category",
    "downstream_bandwidth_cat",
    "upstream_bandwidth_cat",
    "signal_strength_cat",
    "ringing_mode_cat",
    "screen_off_timeout_cat_sec",
    "wifi_notify_risk_flag",
    "animation_scale_cat",
    "ringer_mode_cat",
    "network_country",
    "health_realtime_screen_off_hours",
    "foreground_time_per_use",
    "foreground_intensity",
    "camera_time_per_use",
    "camera_intensity",
    "wifi_scan_intensity",
    "wifi_scan_time_per_scan",
    "system_user_cpu_ratio",
    "wifi_rx_tx_packet_ratio",
    "mobile_rx_tx_packet_ratio",
    "wifi_rx_tx_ratio",
    "mobile_download_upload_ratio",
    "wifi_full_lock_hours",
    "mobile_total_activity",
    "mobile_rx_tx_ratio",
    "mobile_upload_share",
    "wifi_total_time",
    "wifi_active_time",
    "wifi_idle_ratio",
    "wifi_activity_score",
    "screen_off_cpu_activity_ratio",
    "screen_off_ratio",
    "system_user_ratio",
    "top_avg_sec",
    "background_avg_sec",
    "cached_avg_sec",
    "vibrator_avg_sec",
    "mobile_radio_avg_sec",
    "proc_measurement_crash_cat",
    "proc_anr_category",
    "device_risk_flag",
    "collection_duration_min",
    "network_quality",
    "storage_free_ratio",
    "activity_intensity",
    "phone_uptime_hours",
    "device_power_score",
    "Network_usage",
    "wifi_data",
    "entertainment_usage_in_hrs",
    "data_usage_mb",
    "battery_health_score",
    "battery_health_scaled",
    "security_settings_score",
    "security_settings_flag",
    "vpn_flag",
    "tech_savvy_flag",
    "tech_savvy_score",
    "font_scale_bin",
    "battery_level_bin",
    "router_type",
    "PhoneSettings_is_power_save_mode",
   
]

In [131]:
df_andr[feature_list].head()

,customerid,Observation_date,os_version_bucket,device_version_age,display_size_bucket,ram_bucket,main_storage_bucket,is_high_main_storage,storage_utilization,has_sd_card,ext_storage_bucket,large_sd_flag,display_resolution,device_display_category,is_foreign_sim,is_foreign_locale,locale_iso_3_country_grouped,signal_strength_bucket,is_tablet,battery_level_pct,hardware_stress_signal,battery_plugged_type,battery_stress_flag,battery_temp_bin,cpu_core_bin,cpu_arch_bin,wifi_speed_bin,phone_uptime_category,downstream_bandwidth_cat,upstream_bandwidth_cat,signal_strength_cat,ringing_mode_cat,screen_off_timeout_cat_sec,wifi_notify_risk_flag,animation_scale_cat,ringer_mode_cat,network_country,health_realtime_screen_off_hours,foreground_time_per_use,foreground_intensity,camera_time_per_use,camera_intensity,wifi_scan_intensity,wifi_scan_time_per_scan,system_user_cpu_ratio,wifi_rx_tx_packet_ratio,mobile_rx_tx_packet_ratio,wifi_rx_tx_ratio,mobile_download_upload_ratio,wifi_full_lock_hours,mobile_total_activity,mobile_rx_tx_ratio,mobile_upload_share,wifi_total_time,wifi_active_time,wifi_idle_ratio,wifi_activity_score,screen_off_cpu_activity_ratio,screen_off_ratio,system_user_ratio,top_avg_sec,background_avg_sec,cached_avg_sec,vibrator_avg_sec,mobile_radio_avg_sec,proc_measurement_crash_cat,proc_anr_category,device_risk_flag,collection_duration_min,network_quality,storage_free_ratio,activity_intensity,phone_uptime_hours,device_power_score,Network_usage,wifi_data,entertainment_usage_in_hrs,data_usage_mb,battery_health_score,battery_health_scaled,security_settings_score,security_settings_flag,vpn_flag,tech_savvy_flag,tech_savvy_score,font_scale_bin,battery_level_bin,router_type,PhoneSettings_is_power_save_mode
0,2974277,2025-06-26,medium,5,tablet,high,medium,0,0.317329,0,no_sd,0,QHD,FHD+ (mid-range),1,1,USA,missing,0,0.336667,0,on_battery,0,warm,Standard,modern_64bit,medium_speed,Short,Medium_bandwidth,Medium_bandwidth,Very_Strong,Silent,Long,0,Normal,Normal,Unknown,0.000000,21398.250000,1369488.0,1361.666667,12255.0,0.0,NaN,0.288422,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,NaN,0.000000,NaN,0.000000,0.243363,NaN,NaN,NaN,200.0,NaN,Stable,Stable,0,241.0,19.516368,0.682671,18.238781,143.250439,0.0,0.0,0.0,0.0,0.00,0.8,0.2,NaN,1,0.0,1.0,1.0,0.9_to_1,30-39,Home_router,0
1,2288423,2025-11-16,new,3,medium_phones,premium,high,1,0.431331,0,no_sd,0,QHD,Tall screens,0,0,PHL,medium_signal,0,0.589167,0,on_battery,0,warm,Standard,modern_64bit,medium_speed,Short,Medium_bandwidth,High_bandwidth,Very_Strong,Normal,Short,0,Normal,Normal,Philippines,77.561295,16877.000000,1367037.0,1809.666667,16287.0,0.0,NaN,NaN,0.736429,NaN,0.597938,NaN,0.0,0.0,NaN,NaN,310.0,310.0,0.0,5.739793,0.046597,0.806618,NaN,158253.000000,NaN,5.100000e+01,200.0,NaN,Stable,Stable,0,177.0,22.083086,0.568669,9.000000,61.153361,0.0,0.0,2431.0,0.0,0.00,0.8,0.2,1.00,1,0.0,1.0,1.0,0.9_to_1,50-59,Home_router,0
2,2892844,2025-09-03,new,2,large_phones,premium,high,1,0.211332,0,no_sd,0,QHD,Tall screens,0,0,PHL,medium_signal,0,0.641667,0,on_battery,0,warm,Standard,modern_64bit,medium_speed,Short,Medium_bandwidth,Medium_bandwidth,Strong,Silent,Very_Short,0,Normal,Silent,Philippines,1.910512,7449.000000,119184.0,NaN,NaN,0.0,NaN,0.432381,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,NaN,0.000000,0.187431,0.378852,NaN,15195.000000,NaN,2.674333e+03,NaN,NaN,Stable,Stable,0,190.0,19.565151,0.788668,12.055475,28.778310,0.0,0.0,0.0,0.0,0.00,0.8,0.2,1.00,1,0.0,1.0,1.0,0.9_to_1,70-79,Home_router,0
3,3146199,2025-07-25,new,3,medium_phones,high,high,1,0.225057,0,no_sd,0,HD,FHD+ (mid-range),0,1,GBR,medium_signal,0,0.376667,0,on_battery,0,normal,Standard,modern_64bit,medium_speed,Short,Medium_bandwidth,High_bandwidth,Unknown,Normal,Very_Short,0,Normal,Normal,Philippines,18.184628,27889.285714,1366575.0,NaN,NaN,0.0,NaN,0.485093,1.099567,1.16,NaN,3.542017,0.0,1092.0,NaN,0.0,47.0,47.0,0.0,3.871201,0.188214,0.703159,0.463966,57765.000000,1001.25,2.098902e+06,100.0,559981.166667,Stable,Stable,0,544.0,21.922817,0.774943,18.071315

In [132]:
len(feature_list), len(set(feature_list))

(89, 89)

In [133]:
df = df_andr[feature_list].copy()
df.reset_index(inplace=True, drop=True)

In [134]:
df.head(2)

,customerid,Observation_date,os_version_bucket,device_version_age,display_size_bucket,ram_bucket,main_storage_bucket,is_high_main_storage,storage_utilization,has_sd_card,ext_storage_bucket,large_sd_flag,display_resolution,device_display_category,is_foreign_sim,is_foreign_locale,locale_iso_3_country_grouped,signal_strength_bucket,is_tablet,battery_level_pct,hardware_stress_signal,battery_plugged_type,battery_stress_flag,battery_temp_bin,cpu_core_bin,cpu_arch_bin,wifi_speed_bin,phone_uptime_category,downstream_bandwidth_cat,upstream_bandwidth_cat,signal_strength_cat,ringing_mode_cat,screen_off_timeout_cat_sec,wifi_notify_risk_flag,animation_scale_cat,ringer_mode_cat,network_country,health_realtime_screen_off_hours,foreground_time_per_use,foreground_intensity,camera_time_per_use,camera_intensity,wifi_scan_intensity,wifi_scan_time_per_scan,system_user_cpu_ratio,wifi_rx_tx_packet_ratio,mobile_rx_tx_packet_ratio,wifi_rx_tx_ratio,mobile_download_upload_ratio,wifi_full_lock_hours,mobile_total_activity,mobile_rx_tx_ratio,mobile_upload_share,wifi_total_time,wifi_active_time,wifi_idle_ratio,wifi_activity_score,screen_off_cpu_activity_ratio,screen_off_ratio,system_user_ratio,top_avg_sec,background_avg_sec,cached_avg_sec,vibrator_avg_sec,mobile_radio_avg_sec,proc_measurement_crash_cat,proc_anr_category,device_risk_flag,collection_duration_min,network_quality,storage_free_ratio,activity_intensity,phone_uptime_hours,device_power_score,Network_usage,wifi_data,entertainment_usage_in_hrs,data_usage_mb,battery_health_score,battery_health_scaled,security_settings_score,security_settings_flag,vpn_flag,tech_savvy_flag,tech_savvy_score,font_scale_bin,battery_level_bin,router_type,PhoneSettings_is_power_save_mode
0,2974277,2025-06-26,medium,5,tablet,high,medium,0,0.317329,0,no_sd,0,QHD,FHD+ (mid-range),1,1,USA,missing,0,0.336667,0,on_battery,0,warm,Standard,modern_64bit,medium_speed,Short,Medium_bandwidth,Medium_bandwidth,Very_Strong,Silent,Long,0,Normal,Normal,Unknown,0.000000,21398.25,1369488.0,1361.666667,12255.0,0.0,NaN,0.288422,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,NaN,0.000000,NaN,0.000000,0.243363,NaN,NaN,NaN,200.0,NaN,Stable,Stable,0,241.0,19.516368,0.682671,18.238781,143.250439,0.0,0.0,0.0,0.0,0.0,0.8,0.2,NaN,1,0.0,1.0,1.0,0.9_to_1,30-39,Home_router,0
1,2288423,2025-11-16,new,3,medium_phones,premium,high,1,0.431331,0,no_sd,0,QHD,Tall screens,0,0,PHL,medium_signal,0,0.589167,0,on_battery,0,warm,Standard,modern_64bit,medium_speed,Short,Medium_bandwidth,High_bandwidth,Very_Strong,Normal,Short,0,Normal,Normal,Philippines,77.561295,16877.00,1367037.0,1809.666667,16287.0,0.0,NaN,NaN,0.736429,NaN,0.597938,NaN,0.0,0.0,NaN,NaN,310.0,310.0,0.0,5.739793,0.046597,0.806618,NaN,158253.0,NaN,51.0,200.0,NaN,Stable,Stable,0,177.0,22.083086,0.568669,9.000000,61.153361,0.0,0.0,2431.0,0.0,0.0,0.8,0.2,1.0,1,0.0,1.0,1.0,0.9_to_1,50-59,Home_router,0


In [135]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 165989 entries, 0 to 165988
Data columns (total 89 columns):
 #   Column                            Non-Null Count   Dtype         
---  ------                            --------------   -----         
 0   customerid                        165989 non-null  int64         
 1   Observation_date                  165989 non-null  datetime64[us]
 2   os_version_bucket                 165989 non-null  str           
 3   device_version_age                165989 non-null  int64         
 4   display_size_bucket               165989 non-null  str           
 5   ram_bucket                        165989 non-null  str           
 6   main_storage_bucket               165989 non-null  str           
 7   is_high_main_storage              165989 non-null  int64         
 8   storage_utilization               165988 non-null  float64       
 9   has_sd_card                       165989 non-null  int64         
 10  ext_storage_bucket                165989 no

In [136]:
df_andr['Observation_date'] = pd.to_datetime(df_andr['Observation_date'])

# Create Table

In [137]:
from google.cloud import bigquery


client = bigquery.Client(project="prj-prod-dataplatform")

dataset_id = "risk_mart"
table_id = f"{android_devise_score_features}"

table_ref = client.dataset(dataset_id).table(table_id)

job_config = bigquery.LoadJobConfig(
    # schema=schema,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

job = client.load_table_from_dataframe(
    df_andr[feature_list],   # ensure only required columns are loaded
    table_ref,
    job_config=job_config
)

job.result()
print(f"Table {dataset_id}.{table_id} created successfully")


Table risk_mart.bscore_android_device_score_feature_list_v25_20260817 created successfully
